In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from transformers import BertTokenizer, DataCollatorForLanguageModeling
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.utils.data import Dataset

# ---- Step 1: Setup ----

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# ---- Step 2: Load and tokenize WikiText-2 ----
#
# raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train", streaming=True)
# raw_dataset = load_dataset("ag_news", split="train[:5%]")

# def tokenize_function(example):
#     return tokenizer(example["text"], return_special_tokens_mask=True)

# tokenized_dataset = raw_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
import requests

# url = "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt"
# text_data = requests.get(url).text.split('\n')

# from transformers import BertTokenizer
# tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# # Tokenize lines
# tokenized_dataset = [tokenizer.encode(line, return_tensors='pt').squeeze(0)
#                    for line in text_data if line.strip()]

# # ---- Step 3: Chunk into blocks ----

# block_size = 64

# def group_texts(examples):
#     concatenated = {k: sum(examples[k], []) for k in examples.keys()}
#     total_length = (len(concatenated["input_ids"]) // block_size) * block_size
#     return {
#         k: [t[i: i + block_size] for i in range(0, total_length, block_size)]
#         for k, t in concatenated.items()
#     }

# # lm_dataset = tokenized_dataset.map(group_texts, batched=True)
# lm_dataset = list(map(group_texts, tokenized_dataset))

def download_wikitext2():
    url = "https://raw.githubusercontent.com/pytorch/examples/master/word_language_model/data/wikitext-2/train.txt"
    text = requests.get(url).text
    return [line.strip() for line in text.split('\n') if line.strip()]

# ==== Dataset Class ====
class WikiTextDataset(Dataset):
    def __init__(self, lines, tokenizer, block_size=64):
        self.examples = []
        self.tokenizer = tokenizer
        length = len(lines)
        pourcentage = 0.1

        for line in lines[:int(length * pourcentage)]:
            tokens = tokenizer.encode(line, add_special_tokens=True)
            for i in range(0, len(tokens), block_size):
                block = tokens[i:i+block_size]
                if len(block) == block_size:
                    self.examples.append(torch.tensor(block))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]

lines = download_wikitext2()
lm_dataset = WikiTextDataset(lines, tokenizer, block_size=64)

# ---- Step 4: DataLoader with MLM Collator ----

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=True, mlm_probability=0.15)
dataloader = DataLoader(lm_dataset, batch_size=8, shuffle=True, collate_fn=collator)

# ---- Step 5: Define the Tiny Teacher Model ----

class TinyTransformerTeacher(nn.Module):
    def __init__(self, d_model=256, vocab_size=30522, max_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads=1, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        B, L = input_ids.shape
        positions = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        attn_output, attn_weights = self.attn(x, x, x, need_weights=True)
        x = self.ffn(attn_output)
        logits = self.lm_head(x)
        return logits, attn_weights

# ---- Step 6: Training Loop ----

model = TinyTransformerTeacher().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(1):
    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits, attn_weights = model(input_ids)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.view(-1), ignore_index=-100)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if step % 10 == 0:
            print(f"Epoch {epoch} | Step {step} | Loss: {loss.item():.4f}")

# torch.save(model.state_dict(), "teacher.pt")
teacher = model

Token indices sequence length is longer than the specified maximum sequence length for this model (695 > 512). Running this sequence through the model will result in indexing errors


Epoch 0 | Step 0 | Loss: 10.3368
Epoch 0 | Step 10 | Loss: 10.3216
Epoch 0 | Step 20 | Loss: 10.2633
Epoch 0 | Step 30 | Loss: 9.9372
Epoch 0 | Step 40 | Loss: 9.2859
Epoch 0 | Step 50 | Loss: 8.9581
Epoch 0 | Step 60 | Loss: 8.3213
Epoch 0 | Step 70 | Loss: 7.4091
Epoch 0 | Step 80 | Loss: 7.4794
Epoch 0 | Step 90 | Loss: 7.7655
Epoch 0 | Step 100 | Loss: 7.2344
Epoch 0 | Step 110 | Loss: 7.0971
Epoch 0 | Step 120 | Loss: 7.3646
Epoch 0 | Step 130 | Loss: 6.7399
Epoch 0 | Step 140 | Loss: 6.8665
Epoch 0 | Step 150 | Loss: 7.7165
Epoch 0 | Step 160 | Loss: 7.3518
Epoch 0 | Step 170 | Loss: 6.9105
Epoch 0 | Step 180 | Loss: 6.7733
Epoch 0 | Step 190 | Loss: 7.7391
Epoch 0 | Step 200 | Loss: 7.1920
Epoch 0 | Step 210 | Loss: 7.0774
Epoch 0 | Step 220 | Loss: 7.8493
Epoch 0 | Step 230 | Loss: 7.0903
Epoch 0 | Step 240 | Loss: 7.3861
Epoch 0 | Step 250 | Loss: 6.7694
Epoch 0 | Step 260 | Loss: 6.9833
Epoch 0 | Step 270 | Loss: 7.4264
Epoch 0 | Step 280 | Loss: 7.3512
Epoch 0 | Step 290 | L

In [12]:
import torch as t
from torch import nn
from transformers import BertModel
from einops.layers.torch import Rearrange
from torch.nn import functional as F
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizer, DataCollatorForLanguageModeling
from datasets import load_dataset
from tqdm import tqdm

def compute_class_weights(labels, num_classes):
    """
    Args:
        labels (Tensor): shape (batch_size,) – class indices
        num_classes (int): total number of classes
    Returns:
        Tensor: weights of shape (num_classes,)
    """
    class_counts = torch.bincount(labels, minlength=num_classes)
    total = class_counts.sum().item()
    weights = total / (class_counts.float() + 1e-8)  # avoid division by zero
    weights = weights / weights.sum()  # normalize
    return weights

class MemoryTree(nn.Module):

    def __init__(
        self,
        leafs : t.Tensor
    ):
        """
        Args:
            leafs (t.Tensor): B, L, D, D
        """
        super().__init__()

        B, L, D, _ = leafs.shape

        assert L % 2 == 0, "The sequence len need to be divided by 2 for tree building"

        self.tree_depth = t.log2(t.tensor(L)).int()

        self.hierarchical_memory = [leafs]

        last_layer = leafs
        # print(len(leafs))

        # Build hierarchical memory
        for level in range(self.tree_depth - 1):

            level_memory = last_layer.reshape(B, L//2**(level + 1), 2, D, D).sum(dim=2)

            self.hierarchical_memory.append(level_memory)
            last_layer = level_memory

        # print(len(self.hierarchical_memory[1][0]))
        # exit()
        assert len(self.hierarchical_memory[-1][0]) == 2, "Invalid head size"

        self.softmax = nn.Softmax(dim=1)

    def oracle(
        self,
        q : t.Tensor,
        expected : t.Tensor = None
    ):
        """
        Args:
            q (t.Tensor): (B, L_k, D)
            expected (t.Tensor): (B, L_k)
        """
        # print(q.shape)

        B, L_k, D = q.shape
        full_loss = 0.0

        if expected != None:
            for query_id in range(L_k):
                for level in range(self.tree_depth - 1, -1, -1):

                    # print(expected[:, query_id].isnan().all(), level)
                    # Oracle predict labels from expected
                    # print(expected[:, query_id], level)
                    labels = expected[:, query_id] // (2 ** level) # B, 1 One label per batch for a specific token
                    # print(labels)
                    # exit()

                    # Predict model answers using full SA for differentiable training
                    # print(q[:, query_id].unsqueeze(1).unsqueeze(2).shape, self.hierarchical_memory[level].shape)
                    contextual_emb = (q[:, query_id].unsqueeze(1).unsqueeze(2) @ self.hierarchical_memory[level].swapdims(-1,-2)).squeeze(2) # (B, D) @ (B, L//2**level, D, D) -> B, L//2**level, D
                    # logits = self.softmax(contextual_emb.mean(-1))
                    # print("ContextualEmb shape", contextual_emb.shape)
                    # print(contextual_emb.shape, q[:, query_id].unsqueeze(-1).shape)
                    logits = (contextual_emb @ q[:, query_id].unsqueeze(-1)).squeeze(-1)
                    # query = q[:, query_id].unsqueeze(1)  # (B, 1, D)

                    # # Compute unnormalized attention output
                    # contextual_emb = (query.unsqueeze(2) @ self.hierarchical_memory[level].swapdims(-1, -2)).squeeze(2)  # (B, N, D)

                    # # Compute normalization term Z = sum of keys projected onto query
                    # Z = (query @ self.hierarchical_memory[level].sum(-1).transpose(-1, -2)).squeeze(1)  # (B, N)

                    # # Normalize the summary vector for each memory slot
                    # normalized_summary = contextual_emb / (Z.unsqueeze(-1) + 1e-6)  # (B, N, D)

                    # # Project onto the query direction again for logits
                    # logits = (normalized_summary @ query.transpose(1, 2)).squeeze(-1)  # (B, N)

                    # print(self.hierarchical_memory[0].shape)
                    num_classes = self.hierarchical_memory[0].shape[1] // (2 ** level)
                    class_weights = compute_class_weights(labels, num_classes).to(logits.device)
                    # print(class_weights.shape, labels, num_classes)
                    # print(logits, labels)
                    # print(logits.shape, labels.shape)
                    # print(logits, labels)
                    loss = F.cross_entropy(logits.squeeze(-1), labels, weight=class_weights)
                    # print(loss)
                    full_loss += loss

            return full_loss

        else:

            batch_idx = t.arange(B, device = q.device)
            all_query_choices = t.empty(B, L_k, device=q.device)
            for query_id in range(L_k):

                choices = t.zeros(B, dtype=t.long, device=q.device)
                for level in range(self.tree_depth - 1, -1, -1):

                    # Only select the memory we need for processing the hard gating
                    # Based on q (B, L_k, D)
                    query = q[:,query_id] # B, D
                    # if choices != 0:
                    # print("Hierarchical_memory[level].shape", self.hierarchical_memory[level].shape)
                    left_embeddings = self.hierarchical_memory[level][batch_idx, choices * 2] # B, This are global choices
                    right_embeddings = self.hierarchical_memory[level][batch_idx, choices * 2 + 1]
                    # B, 1, D, D
                    # 8, D, D

                    # print((left_embeddings -  right_embeddings).sum())
                    # return

                    # print("query.shape, left_embeddings.shape", query.shape, left_embeddings.shape)
                    # print(query.unsqueeze(1).shape, left_embeddings.shape)
                    # print(query.unsqueeze(1).shape, right_embeddings.shape)
                    # print((query @ left_embeddings).shape)
                    # print(query.unsqueeze(1).shape, left_embeddings.swapdims(-1,-2).shape)
                    left_contextual_emb = (query.unsqueeze(1) @ left_embeddings.swapdims(-1,-2) @ query.unsqueeze(-1)).reshape(B)
                    right_contextual_emb = (query.unsqueeze(1) @ right_embeddings.swapdims(-1,-2) @ query.unsqueeze(-1)).reshape(B)

                    # # print("choices.shape", choices.shape)
                    # # print("left_contextual_emb.shape", left_contextual_emb.shape)
                    # # print(left_contextual_emb < right_contextual_emb)
                    choices = (choices * 2) + (left_contextual_emb < right_contextual_emb)
                    # Compute contextualized embeddings
                    # left_proj = (query.unsqueeze(1) @ left_embeddings.swapdims(-1, -2)).squeeze(1)  # (B, D)
                    # right_proj = (query.unsqueeze(1) @ right_embeddings.swapdims(-1, -2)).squeeze(1)
                    # print(left_proj.shape, right_proj.shape)

                    # Compute normalization factors Z
                    # Z_left = (query @ left_embeddings.sum(-1).transpose(-1, -2)).squeeze(1) + 1e-6  # (B,)
                    # Z_right = (query @ right_embeddings.sum(-1).transpose(-1, -2)).squeeze(1) + 1e-6
                    # Z_left = torch.einsum('bd,bdd->b', query, left_embeddings) + 1e-6
                    # Z_right = torch.einsum('bd,bdd->b', query, right_embeddings) + 1e-6

                    # Normalize
                    # print("Zleft", Z_left.shape, "ZRight", Z_right.shape)
                    # print(query.shape)
                    # print((left_proj / Z_left.unsqueeze(-1)).shape)
                    # left_norm = ((left_proj / Z_left.unsqueeze(-1)).unsqueeze(1) @ query.unsqueeze(-1)).reshape(B)  # (B, 1)
                    # right_norm = ((right_proj / Z_right.unsqueeze(-1)).unsqueeze(1) @ query.unsqueeze(-1)).reshape(B)

                    # print(left_norm.shape, right_norm.shape)
                    # Compare
                    # choices = (choices * 2) + (left_norm.squeeze(-1) < right_norm.squeeze(-1))
                    # print(choices.shape)

                # all_query_choices.append(choices)
                all_query_choices[:,query_id] = choices

            # print(len(all_query_choices))

            return all_query_choices


class HierarchicalLinearAttention(nn.Module):

    def __init__(
        self,
        d_model : int
    ):
        super().__init__()

        self.W_kv = nn.Linear(d_model, 2 * d_model, bias = False)
        self.W_q = nn.Linear(d_model, d_model, bias=False)

        self.bert_embeddings = BertModel.from_pretrained("bert-base-uncased").get_input_embeddings()
        self.rearrange = Rearrange('b l (k d) -> k b l d', k=2)

    def build_tree(
        self,
        context : t.Tensor
    ):
        B, L = context.shape
        context_embeddings = self.bert_embeddings(context)
        D = context_embeddings.shape[-1]
        key, values = self.rearrange(self.W_kv(context_embeddings)) # 2, B, L, D
        key = key
        values = values

        leafs = key.unsqueeze(-1) @ values.unsqueeze(-2)

        # When building the Tree object, remember to detach() gradients
        return MemoryTree(leafs)

    def forward(
        self,
        query : t.Tensor,
        memory_tree : MemoryTree,
        expected : t.Tensor = None
    ):
        """
        Args:
            query (t.Tensor): (B, L_k)
            memory_tree (MemoryTree):
            returns a loss in case of training and prediction in case of eval
        """

        query_emb = self.bert_embeddings(query)
        # print(query_emb.shape)
        query_emb = self.W_q(query_emb)
        # print(self.W_q.weight.sum())
        # print(self.W_q.weight.var())
        return memory_tree.oracle(query_emb, expected)


class TinyTransformerTeacher(nn.Module):
    def __init__(self, d_model=256, vocab_size=30522, max_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads=1, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        B, L = input_ids.shape
        positions = torch.arange(L, device=input_ids.device).unsqueeze(0).expand(B, L)
        x = self.token_emb(input_ids) + self.pos_emb(positions)
        attn_output, attn_weights = self.attn(x, x, x, need_weights=True)
        x = self.ffn(attn_output)
        logits = self.lm_head(x)
        return logits, attn_weights


def compute_accuracy(predicted_indices, true_indices):
    correct = 0
    total = 0
    for pred, true in zip(predicted_indices, true_indices):
        print(pred, true)
        correct += (pred == true).sum().item()
        total += len(true)
    return correct / total if total > 0 else 0.0

# def compute_accuracy(predicted_indices, true_indices):
#     correct = 0
#     total = 0
#     for pred, true in zip(predicted_indices, true_indices):
#         # print(pred.device, true.device)
#         print(pred, true)
#         correct += t.norm(pred - true).sum().item()
#         total += len(true)
#         # print(len(true), correct)
#     return correct / total


if __name__ == "__main__":

    teacher.eval()

    # === Initialize HLA ===
    hla = HierarchicalLinearAttention(d_model=768).cuda()
    hla.train()

    optimizer = torch.optim.Adam(hla.parameters(), lr=1e-3)

    # === Training loop ===
    for epoch in range(3):
        for step, batch in enumerate(tqdm(dataloader)):
            context_ids = batch["input_ids"].cuda()

            with torch.no_grad():
                _, attn_weights = teacher(context_ids)  # (B, L, L)
                top1_indices = attn_weights.argmax(dim=-1)  # (B, L)

            # print(context_ids.shape)
            # exit()
            memory_tree = hla.build_tree(context_ids)
            # print(top1_indices.shape)
            loss = hla.forward(
                query=context_ids,
                memory_tree=memory_tree,
                expected=top1_indices
            )

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            # --- Evaluate top-1 match ---
            with torch.no_grad():
                predictions = memory_tree.oracle(hla.W_q(hla.bert_embeddings(context_ids)))
                # predictions = memory_tree.oracle(hla.bert_embeddings(context_ids))
                acc = compute_accuracy(predictions, top1_indices.unsqueeze(1))

            print(f"[Epoch {epoch} Step {step}] Loss: {loss.item():.4f} | Acc: {acc:.4f}")

  0%|          | 1/374 [00:04<26:09,  4.21s/it]

tensor([ 8.,  8.,  8.,  8.,  6.,  8.,  8.,  3.,  8.,  8.,  8.,  8.,  8.,  8.,
         8.,  8.,  8.,  8.,  8.,  0.,  8.,  8.,  8.,  8.,  8.,  8.,  8.,  8.,
         8.,  8.,  8.,  8.,  8., 53., 62.,  6.,  8.,  8.,  8.,  8.,  8.,  8.,
         8., 39.,  8.,  8.,  8.,  8.,  8.,  8.,  8., 46.,  8.,  8.,  8.,  8.,
         8.,  8.,  8.,  8.,  8.,  8., 46., 62.], device='cuda:0') tensor([[46, 46, 38, 46,  3, 26,  9, 26, 46, 46, 46, 10, 46, 48, 37, 46, 46, 48,
         46, 63, 10, 48, 46, 46, 26, 26, 46, 46, 10, 46, 56, 46, 48,  3, 40,  3,
         10, 46, 10,  9, 46,  9, 56,  9, 46, 56, 46,  5, 46, 46, 63, 46, 46, 46,
         46, 46, 46, 46, 56, 20, 56, 46, 46, 46]], device='cuda:0')
tensor([40., 40., 40., 63., 40., 63., 63., 40., 40., 40., 40., 40., 40., 40.,
        40., 40., 40., 40., 63., 24., 40., 40., 40., 40., 63., 40., 40., 40.,
        40., 63., 40., 40., 32., 40., 63., 40., 40., 40., 24., 14., 63., 63.,
        56., 20., 40., 63., 40., 40., 40., 40., 40., 63., 40., 40., 24., 40.,

  1%|          | 2/374 [00:08<26:03,  4.20s/it]

tensor([16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 43., 16., 16.,
        16., 16., 43., 43., 16., 16., 16., 16., 16., 43., 16., 16., 16., 16.,
        16., 16., 16., 16., 16., 43., 16., 16., 16., 16., 16., 16., 16., 16.,
        16., 43., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16.,
        16., 16., 16., 16., 16., 43., 16., 43.], device='cuda:0') tensor([[16, 43, 10, 41, 16, 23, 23, 43, 61, 34, 23, 43, 20, 43, 23, 43, 11, 43,
         43, 11, 23, 43, 43, 43, 23, 23, 11, 23, 43, 23, 62,  2,  7, 43, 23, 16,
         41, 43, 43, 58, 23, 43, 16, 43, 43,  6, 23, 56, 23, 61, 20, 11, 61, 23,
         16, 23, 43, 23, 58, 23, 43, 43, 23, 36]], device='cuda:0')
tensor([32., 48., 12., 32., 12., 48., 48., 32., 12., 12., 12., 12., 12., 12.,
        12., 48., 12., 12., 32., 48., 48., 32., 12., 48., 32., 48., 12., 32.,
        12., 12., 32., 48., 12., 12., 12., 48., 32., 12., 48., 48., 32., 48.,
        48., 12., 12., 12., 12., 48., 12., 12., 12., 32., 12., 48., 32., 48.,

  1%|          | 3/374 [00:12<25:58,  4.20s/it]

...

tensor([30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30.,
        30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30.,
        30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30.,
        30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30., 30.,
        30., 30., 30., 30., 30., 30., 30., 30.], device='cuda:0') tensor([[28, 21, 28, 26, 50, 25, 28, 21, 15, 15,  6, 31, 31, 61, 61, 28, 15,  2,
         28, 56, 28, 28, 61, 21, 26, 21, 28,  2, 61, 50, 61, 28, 21, 61, 40, 61,
         61, 50, 23,  8, 28, 28, 35, 31, 61, 15, 21, 28, 28, 50, 28, 50, 61, 21,
         28, 28, 15, 50, 28, 50, 28, 28,  6, 15]], device='cuda:0')
tensor([59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59.,
        59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59.,
        59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59., 59.,
        59., 59., 59., 59., 61., 59., 59., 59., 59., 59., 59., 59., 59., 59.,

 22%|██▏       | 82/374 [05:45<20:28,  4.21s/it]

tensor([36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36.,
        36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36.,
        36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36.,
        36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36., 36.,
        36., 36., 36., 36., 36., 36., 36., 36.], device='cuda:0') tensor([[ 8, 36, 17, 36, 20, 36, 20, 53, 36, 47, 56, 36, 14, 36,  4, 36, 36, 39,
         53, 20, 36, 36, 37, 36, 36, 19, 36, 20, 20, 36, 16, 20, 36, 36, 41, 37,
         36, 36, 20, 36, 37, 36, 36, 36, 20, 56,  8, 36, 20, 20, 20, 53, 36, 36,
         36, 37, 36, 20,  2, 20, 20, 20, 20, 20]], device='cuda:0')
tensor([57., 57., 57., 57., 57., 57., 57., 57., 57., 57., 57., 57., 57., 57.,
        36., 57., 57., 57., 57., 57., 36., 36., 57., 57., 57., 57., 57., 57.,
        57., 57., 57., 36., 57., 36., 57., 57., 36., 57., 36., 57., 36., 57.,
        57., 57., 57., 36., 57., 57., 57., 57., 36., 57., 57., 57., 57., 36.,

 22%|██▏       | 82/374 [05:49<20:43,  4.26s/it]


KeyboardInterrupt: 